# 03 特徵工程

| | 內容 |
|---|---|
| 輸入 | `data/processed/targets.parquet`（01 的輸出） |
| 輸出 | 無（特徵在訓練時依預測起點即時組裝；本 notebook 呈現特徵內容與防洩漏檢查） |

每個預測步長（h = 1、2、3 天）各一份特徵矩陣。落後特徵依 h 建構，確保只用預測起點以前的資料；
下方以**擾動測試**驗證：把某一天的目標改成極端值，所有不該看到它的列都不得改變。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config").is_dir():   # 找到專案根目錄（notebook 可從任何位置執行）
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import os
import polars as pl
from src.logging_setup import setup_logging

os.chdir(ROOT)
setup_logging(stage="notebook_03", level="WARNING")
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(60)

polars.config.Config

In [2]:
from config import paths, settings
from src.data import external
from src.features import builder, lag

daily = pl.read_parquet(paths.TARGETS_FILE)
calendar_df = external.load_calendar()
rules = external.load_price_period_rules()
print("特徵開關：氣象", settings.ENABLE_TIER1_WEATHER, "／太陽光電", settings.ENABLE_WINDY,
      "／農曆", settings.ENABLE_LUNAR_FEATURES, "／特殊日期", settings.ENABLE_SPECIAL_DAYS)

特徵開關：氣象 True ／太陽光電 True ／農曆 True ／特殊日期 True


In [3]:
features = {}
for horizon in range(1, settings.PREDICT_HORIZON_DAYS + 1):
    features[horizon] = builder.build_features(daily, calendar_df, rules, horizon)
pl.DataFrame([
    {"h": h, "列數": f.height, "欄數": f.width, "特徵數": len(builder.feature_names(f))}
    for h, f in features.items()
])

2026-10-01 16:22:06 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:08 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:09 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


h,列數,欄數,特徵數
i64,i64,i64,i64
1,1004,167,120
2,1004,161,114
3,1004,161,114


## 特徵清單（h = 1）

依欄名前綴分組。`builder.NON_FEATURE_COLUMNS` 列出的欄位（識別欄、字串欄、由當日標籤算出的
診斷欄）與 `settings.EXCLUDED_FEATURES` 不會進入模型。

In [4]:
import re

names = builder.feature_names(features[1])
RULES = [                                   # 依序比對，第一個符合的類別勝出
    ("落後（自迴歸）", r"_(lag\d+|ma\d+|med\d+|std\d+|daytype_ma\d+)$"),
    ("氣象", r"^w_"),
    ("電價時段", r"^(day|night)_(share|price)|^is_offpeak_day$"),
    ("颱風", r"^is_typhoon"),
    ("農曆與節氣", r"lunar|solar_term"),
    ("特殊日期（行政機關辦公日曆）", r"^is_(holiday|vacation|exam|sports_event|shopping_fest)$|_admin$"),
    ("天文", r"^(sunrise|sunset|solar_noon|noon_elevation|day_length)"),
    ("太陽光電預測（Windy）", r"^pv_"),
    ("日曆與國定假日", r".*"),
]
def group_of(name: str) -> str:
    return next(label for label, pattern in RULES if re.search(pattern, name))
groups = pl.DataFrame({"特徵": names}).with_columns(
    pl.col("特徵").map_elements(group_of, return_dtype=pl.Utf8).alias("類別"))
groups.group_by("類別", maintain_order=True).agg(pl.len().alias("個數"), pl.col("特徵").head(6).alias("例"))

類別,個數,例
str,u32,list[str]
"""日曆與國定假日""",11,"[""weekday"", ""is_summer"", … ""is_day_before_holiday""]"
"""電價時段""",10,"[""day_share_半尖峰"", ""day_share_尖峰"", … ""night_share_半尖峰""]"
"""天文""",6,"[""sunrise_min"", ""sunset_min"", … ""sunset_minus_night_start""]"
"""落後（自迴歸）""",60,"[""p_day_lag1"", ""t_day_lag1"", … ""ramp_down_lag1""]"
"""氣象""",20,"[""w_新北_day_tmax"", ""w_臺北_day_tmax"", … ""w_新北_day_tmin""]"
"""太陽光電預測（Windy）""",2,"[""pv_14"", ""pv_sum""]"
"""農曆與節氣""",5,"[""lunar_month"", ""lunar_day"", … ""days_since_solar_term""]"
"""特殊日期（行政機關辦公日曆）""",6,"[""is_holiday"", ""is_vacation"", … ""holiday_day_index_admin""]"


In [5]:
groups

特徵,類別
str,str
"""weekday""","""日曆與國定假日"""
"""is_summer""","""日曆與國定假日"""
"""day_share_半尖峰""","""電價時段"""
"""day_share_尖峰""","""電價時段"""
"""day_share_週六半尖峰""","""電價時段"""
"""day_share_離峰""","""電價時段"""
"""day_price_switches""","""電價時段"""
"""night_share_半尖峰""","""電價時段"""
"""night_share_尖峰""","""電價時段"""


## 洩漏檢查

`lag.assert_no_leakage`：把某一天的目標值改成極端值、重算特徵，確認所有「目標日 ≤ 該日 + h − 1」
的列都沒有變化。有變化就代表那些列看到了不該看到的資料，會直接拋錯。

In [6]:
for horizon, frame in features.items():
    lag.assert_no_leakage(frame, daily, horizon)
    print(f"h = {horizon}：通過")

h = 1：通過
h = 2：通過
h = 3：通過


## 特徵選擇

刪除任何變數都必須有數據依據。做法在執行前就決定好：以資料截止日以前的資料，
依上線設定訓練全部 18 個模型（量值 4 目標 × 3 horizon、學習式時刻 2 目標 × 3 horizon），
統計每個特徵被用來分裂的次數；**所有模型都是 0 次**的特徵列為刪除候選，再以回測確認。

`settings.EXCLUDED_FEATURES` 目前已排除的特徵，這裡暫時放回來一起統計，才看得到它們的證據。

In [7]:
from config import settings
from src.evaluation import backtest, importance
from src.workflow import temporary_settings

with temporary_settings(EXCLUDED_FEATURES=()):
    usage = importance.split_usage()
print(f"特徵 {usage.height} 個；所有模型都沒用過的 {usage.filter(pl.col('splits') == 0).height} 個")
usage.filter(pl.col("splits") == 0).select("feature", "models_used", "splits", "gain")

2026-10-01 16:22:12 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:13 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:15 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:17 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:25 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


2026-10-01 16:22:32 | WARNING  | src.data.checks | Windy：完全重複 122 列，已刪除


特徵 126 個；所有模型都沒用過的 5 個


feature,models_used,splits,gain
str,u32,i64,f64
"""is_typhoon_impacting""",0,0,0.0
"""is_lunar_new_year_eve""",0,0,0.0
"""is_lunar_festival""",0,0,0.0
"""is_typhoon_day""",0,0,0.0
"""night_price_switches""",0,0,0.0


In [8]:
usage.select("feature", "models_used", "splits", "gain").head(20)   # 使用最少的 20 個

feature,models_used,splits,gain
str,u32,i64,f64
"""is_typhoon_impacting""",0,0,0.0
"""is_lunar_new_year_eve""",0,0,0.0
"""is_lunar_festival""",0,0,0.0
"""is_typhoon_day""",0,0,0.0
"""night_price_switches""",0,0,0.0
"""is_summer""",3,3,5.32215
"""is_exam""",3,4,5.849365
"""is_sports_event""",3,6,23.15928
"""holiday_day_index""",6,12,41.45447


`settings.EXCLUDED_FEATURES` 的 6 個特徵是在開發期（資料到 2026-06-30）選出的：當時這 6 個在全部模型中都是 0 次。
以全部資料重算，`is_exam` 有少數幾次分裂，其餘 5 個仍是 0 次；排除清單維持不變。

回測確認（開發期 78 窗）：刪除這 6 個特徵後，逐窗分數**完全相同**
（LightGBM 本來就無法用它們分裂：非預設值的樣本太少）。依「平手選較簡單者」刪除。

In [9]:
backtest.experiment_table("*_4f_drop_zero_split_tuning").select(
    "total_score", "同季節", "差值", "同季節差值", "對照")

total_score,同季節,差值,同季節差值,對照
f64,f64,f64,f64,str
1.064475,0.760738,0.0,0.0,"""20260928_101300_4a_A1_noUV_tuning"""
